In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
%run ./_local_config

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()

# Decode bytes to text, split into lines, parse each line as its own JSON object
# (each line here is one full API page response, not one sales record)
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

# Each page has a "value" key containing a list of actual sales records —
# flatten all pages into a single list of records
all_records = []
for page in pages:
    all_records.extend(page["value"])

bronze = pd.DataFrame(all_records)

print(bronze.shape)
print(bronze.columns.tolist())
bronze.head()

(596726, 23)
['itemNo', 'postingDate', 'entryType', 'documentType', 'locationCode', 'quantity', 'subDivision', 'salesPersonCode', 'itemDescription', 'costAmountActual', 'salesAmountActual', 'locationDescription', 'salespersonName', 'brandCode', 'brandDescription', 'itemCategoryCode', 'itemCategory2', 'itemCategory3', 'itemCategory4', 'auxiliaryIndex1', 'auxiliaryIndex2', 'auxiliaryIndex3', 'auxiliaryIndex4']


,itemNo,postingDate,entryType,documentType,locationCode,quantity,subDivision,salesPersonCode,itemDescription,costAmountActual,salesAmountActual,locationDescription,salespersonName,brandCode,brandDescription,itemCategoryCode,itemCategory2,itemCategory3,itemCategory4,auxiliaryIndex1,auxiliaryIndex2,auxiliaryIndex3,auxiliaryIndex4
0,PMFS65,2021-04-01,Sale,Sales Shipment,202CWHR_10,-10.0,202APP,SJ-001699,BATTERY EXIDE POWER MF 12V-65AH,-86754.80,121670.39,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,BATTERY,SUV,,,883,27,PMFS65,BRAND
1,MF105D31R,2021-04-01,Sale,Sales Shipment,202CWHR_10,-4.0,202ULT,WEI,BATTERY EXIDE ULTRA 12V-90AH,-47073.44,68859.28,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,W. E. I,EXIDE,EXIDE,BATTERY,LORRY,,,884,27,MF105D31R,BRAND
2,MF105D31R,2021-04-01,Sale,Sales Shipment,202CWHR_10,-6.0,202ULT,WEI,BATTERY EXIDE ULTRA 12V-90AH,-70610.16,103288.92,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,W. E. I,EXIDE,EXIDE,BATTERY,LORRY,,,885,27,MF105D31R,BRAND
3,3WBMF,2021-04-01,Sale,Sales Shipment,202CWHR_10,-5.0,202APP,SJ-001699,BATTERY EXIDE MF 3WH 12V-35AH,-22663.50,30756.25,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,BATTERY,THREE WHEELER,,,886,27,3WBMF,BRAND
4,PMF38B20L,2021-04-01,Sale,Sales Shipment,202CWHR_10,-5.0,202APP,SJ-001699,BATTERY EXIDE POWER MF 12V-35AH,-23355.75,34389.14,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,BATTERY,CAR,,,887,27,PMF38B20L,BRAND


In [0]:
print(bronze.shape)  # expect something in the hundreds of thousands, matching your earlier ~596K count
print(bronze["itemCategoryCode"].unique())

(596726, 23)
['BATTERY' 'MOTOR CYCLE' 'INDUSTRIAL BAT' 'BAT-OTHERS' 'TYRES' 'FUEL'
 'HYBRID CENTER' 'ACCESSORY' '']


In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(bronze)
print(silver.shape)
silver.head()

(425051, 14)


,posting_date,item_no,itemCategoryCode,vehicle_type,location_code,location_description,sales_person_code,salesperson_name,brand_code,brand_description,documentType,units_sold,costAmountActual,salesAmountActual
0,2021-04-01,PMFS65,BATTERY,SUV,202CWHR_10,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,SJ-001699,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,Sales Shipment,10.0,-86754.80,121670.39
1,2021-04-01,MF105D31R,BATTERY,LORRY,202CWHR_10,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,WEI,W. E. I,EXIDE,EXIDE,Sales Shipment,4.0,-47073.44,68859.28
2,2021-04-01,MF105D31R,BATTERY,LORRY,202CWHR_10,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,WEI,W. E. I,EXIDE,EXIDE,Sales Shipment,6.0,-70610.16,103288.92
3,2021-04-01,3WBMF,BATTERY,THREE WHEELER,202CWHR_10,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,SJ-001699,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,Sales Shipment,5.0,-22663.50,30756.25
4,2021-04-01,PMF38B20L,BATTERY,CAR,202CWHR_10,CENTRAL WAREHOUSE RATHMALANA - EXIDE PRODUCTS,SJ-001699,KONARA MUDIYANSELAGE LAKSHAMAN,EXIDE,EXIDE,Sales Shipment,5.0,-23355.75,34389.14


In [0]:
unique_categories = sorted(silver["itemCategoryCode"].dropna().unique())
print(f"Found {len(unique_categories)} unique item categories")
print(unique_categories)

Found 1 unique item categories
['BATTERY']


In [0]:
vehicle_counts = silver["vehicle_type"].value_counts(dropna=False)
print(vehicle_counts)

vehicle_type
CAR              137236
LORRY            111637
SUV               73433
THREE WHEELER     52836
BUS               39188
TRAILERS           4733
VAN                3362
BOATS              2471
INVERTER            155
Name: count, dtype: int64


In [0]:
unique_vehicle_types = sorted(silver["vehicle_type"].dropna().unique())

vehicle_type_lookup = pd.DataFrame({
    "vehicle_type_id": [f"V{i}" for i in range(1, len(unique_vehicle_types) + 1)],
    "vehicle_type": unique_vehicle_types
})

print(vehicle_type_lookup)

  vehicle_type_id   vehicle_type
0              V1          BOATS
1              V2            BUS
2              V3            CAR
3              V4       INVERTER
4              V5          LORRY
5              V6            SUV
6              V7  THREE WHEELER
7              V8       TRAILERS
8              V9            VAN


In [0]:
brand_counts = silver.groupby(["brand_code", "brand_description"]).size().reset_index(name="count")
brand_counts = brand_counts.sort_values("count", ascending=False)
print(brand_counts.to_string())

  brand_code brand_description   count
0      EXIDE             EXIDE  425051


In [ ]:
import json

# Convert DataFrame to JSON records, handling datetime serialization
silver_json = silver.to_json(orient="records", date_format="iso", lines=False)

blob_client = blob_service.get_blob_client(
    container="silver",
    blob="erp/battery/battery_clean.json"
)
blob_client.upload_blob(silver_json, overwrite=True)

print(f"Saved {len(silver)} rows to silver/erp/battery/battery_clean.json")